<a href="https://colab.research.google.com/github/hungvuongminh-cell/AI-Testing-Learning/blob/main/0007_Automated__Multi_Model_Evaluation_Framework.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# 1. Cài đặt thư viện cần thiết
!pip install langchain-groq gspread google-auth
from IPython.display import clear_output
# ... sau khi chạy xong các lệnh !pip ...
clear_output()

In [21]:
import os, time, textwrap, gspread
from google.colab import userdata, auth
from google.auth import default
from langchain_groq import ChatGroq

# --- 2. XÁC THỰC & KẾT NỐI ---
auth.authenticate_user()
creds, _ = default()
client = gspread.authorize(creds)

spreadsheet_id = '154YYMTztaUBNVHstieW7SMZW-f4hmodiMUjcFcXEoFg'
spreadsheet = client.open_by_key(spreadsheet_id)
sheet1 = spreadsheet.sheet1
sheet2 = spreadsheet.worksheet("sheet2")

groq_api_key = userdata.get('groq_api')

# --- 3. LẤY DỮ LIỆU ĐẦU VÀO ---
model_names = [m.strip() for m in sheet1.acell('A2').value.split(',') if m.strip()]
judge_model_name = sheet1.acell('B2').value or "llama-3.3-70b-versatile"
test_cases = sheet1.get_all_values()[1:]

print(f"🚀 Bắt đầu Test: {len(model_names)} models | {len([r for r in test_cases if r[2]])} kịch bản.")
print("📊 Kết quả sẽ được ghi vào Sheet2...\n")

# --- 4. VÒNG LẶP TEST & XUẤT DỮ LIỆU ---
for i, row in enumerate(test_cases, 1):
    if len(row) < 3 or not row[2].strip(): continue

    question, sys_test, sys_judge = row[2], row[3], row[4]
    print(f"➡️ Đang xử lý Test #{i}: {question[:50]}...")

    for model_name in model_names:
        try:
            # 1. Gọi Model Test
            llm_test = ChatGroq(temperature=0, model_name=model_name, groq_api_key=groq_api_key)
            answer = llm_test.invoke([("system", sys_test or "Assistant"), ("human", question)]).content

            # 2. Gọi Model Judge
            llm_judge = ChatGroq(temperature=0, model_name=judge_model_name, groq_api_key=groq_api_key)
            judge_res = llm_judge.invoke([("system", sys_judge or "Judge"), ("human", f"Q: {question}\nA: {answer}")]).content

            # 3. Ghi dữ liệu vào Sheet2 (Dòng mới cuối cùng)
            # Cấu trúc: [Câu hỏi, Model, Câu trả lời AI, Đánh giá của Judge]
            sheet2.append_row([question, model_name, answer, judge_res])

            print(f"   ✅ Đã ghi kết quả cho {model_name.split('/')[-1]}")
            time.sleep(2) # Tránh Rate Limit

        except Exception as e:
            print(f"   ❌ Lỗi tại model {model_name}")

print("\n✨ HOÀN THÀNH! Bạn hãy kiểm tra Sheet2 để xem kết quả.")

🚀 Bắt đầu Test: 3 models | 2 kịch bản.
📊 Kết quả sẽ được ghi vào Sheet2...

➡️ Đang xử lý Test #1: "Tôi là một nhân viên Marketing truyền thống với 5...
   ✅ Đã ghi kết quả cho gpt-oss-120b
   ✅ Đã ghi kết quả cho gpt-oss-20b
   ✅ Đã ghi kết quả cho llama-3.1-8b-instant
➡️ Đang xử lý Test #2: "Công ty tôi vừa áp dụng quy định: Tất cả báo cáo ...
   ✅ Đã ghi kết quả cho gpt-oss-120b
   ✅ Đã ghi kết quả cho gpt-oss-20b
   ✅ Đã ghi kết quả cho llama-3.1-8b-instant

✨ HOÀN THÀNH! Bạn hãy kiểm tra Sheet2 để xem kết quả.
